In [50]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [51]:
gilt_eps = 1e-4 #from the paper for this chi
chi = 10
cg_eps = 1e-10
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 0,
	"rotate" => true,
)
Jratio = 1.0

relT=1.0
#do 3 steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, 3, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[4], accepted_elements, _ = fix_discrete_gauge(traj[4]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

Newton iterations below (I interrupted the code after a few iterations, but in previous runs I saw it converge)

In [52]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

A[1] = traj[4]
for i in 1:20
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    RA = gilt(A[i], accepted_elements[i], gilt_pars);
    println("i=",i) 
    println("||R(A[i])-A[i]||= ", embedded_distance(RA, A[i]))
    flush(stdout)
    #deltaA[i] = newton_correction(A[i], 5, accepted_elements[i], gilt_pars);
    deltaA[i] = newton_correction_with_iterations_fixed(A[i], 5, accepted_elements[i], gilt_pars);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    A[i+1] = A[i] + deltaA[i]
end

i=1
||R(A[i])-A[i]||= 0.03024339787576557
Dict{Any, Any}((1, "N") => 38, (1, "W") => 24, (1, "S") => 41, (1, "E") => 25, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (1.2832725613868029e-48, 4.597894810275099e-31, 1.4911230694335404e-31, 3.261210980756246e-20, 4.110103985105995e-17, 2.0371988349080272e-16, 2.0371988349080272e-16, 9.579874788502246e-17, 9.579874788502246e-17, 1.3651952970510674e-16, 1.3651952970510674e-16)
└ *  number of operations = 43


EIGENVALUES (INITIAL):
1.997298058831637 + 0.0im
-0.9162211397290262 + 0.0im
-0.9131293477255296 + 0.0im
0.4421689663728989 + 0.0im
-0.3345531875019054 + 0.0im
-5.343441873978743e-5 + 0.31204126895423656im
-5.343441873978743e-5 - 0.31204126895423656im
-0.07263109823285557 + 0.29393259128903737im
-0.07263109823285557 - 0.29393259128903737im
0.07263259246913056 + 0.2938284763662346im
0.07263259246913056 - 0.2938284763662346im
||deltaA[i]||= 0.024245861307882314
i=2
||R(A[i])-A[i]||= 0.006849404556926269
Dict{Any, Any}((1, "N") => 45, (1, "W") => 28, (1, "S") => 48, (1, "E") => 29, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (2.540856699542103e-46, 2.6217045734174034e-31, 2.597771365754423e-30, 1.1366526191460728e-20, 2.4447879485767e-17, 1.1132506036361879e-15, 1.1132506036361879e-15, 8.588094913386372e-16, 8.588094913386372e-16, 8.557918923112126e-16, 8.557918923112126e-16)
└ *  number of operations = 43


EIGENVALUES (INITIAL):
2.005192290414929 + 0.0im
-0.9355299768202254 + 0.0im
-0.9304462266175683 + 0.0im
0.44361263080684643 + 0.0im
-0.354355248628453 + 0.0im
-4.596968758185864e-5 + 0.31650605296656464im
-4.596968758185864e-5 - 0.31650605296656464im
0.06456239193916605 + 0.29530512928955654im
0.06456239193916605 - 0.29530512928955654im
-0.06453876506750156 + 0.29522088443824906im
-0.06453876506750156 - 0.29522088443824906im
||deltaA[i]||= 0.00530862809463107
i=3
||R(A[i])-A[i]||= 0.001632018441410322
Dict{Any, Any}((1, "N") => 46, (1, "W") => 29, (1, "S") => 50, (1, "E") => 30, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


...

EIGENVALUES (INITIAL):
2.007424300019897 + 0.0im
-0.9409845059590775 + 0.0im
-0.9361141686360289 + 0.0im
0.4442513289440716 + 0.0im
-0.36276142917292575 + 0.0im
||deltaA[i]||= 4.573556746725965e-12
i=19
||R(A[i])-A[i]||= 1.7247800610698296e-12
Dict{Any, Any}((1, "N") => 47, (1, "W") => 30, (1, "S") => 51, (1, "E") => 30, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 2 iterations:
│ *  5 eigenvalues converged
│ *  norm of residuals = (7.137319444846462e-34, 2.056145217355908e-19, 1.1548744682631761e-21, 4.7811563748282294e-14, 9.673038943505876e-13)
└ *  number of operations = 34


EIGENVALUES (INITIAL):
2.0074245676495943 + 0.0im
-0.9409843766693862 + 0.0im
-0.9361144358400826 + 0.0im
0.44425103114444303 + 0.0im
-0.36276023812726327 + 0.0im
||deltaA[i]||= 1.8585959620659776e-12
i=20
||R(A[i])-A[i]||= 7.342728369821639e-13
Dict{Any, Any}((1, "N") => 47, (1, "W") => 30, (1, "S") => 51, (1, "E") => 30, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 2 iterations:
│ *  5 eigenvalues converged
│ *  norm of residuals = (8.877070195640989e-35, 1.3591992255041517e-22, 4.717576521836127e-22, 4.278035012713573e-15, 2.4593393616491933e-13)
└ *  number of operations = 34


EIGENVALUES (INITIAL):
2.0074243211635108 + 0.0im
-0.940984280287024 + 0.0im
-0.9361146649634611 + 0.0im
0.4442510678751726 + 0.0im
-0.3627600254654668 + 0.0im
||deltaA[i]||= 7.426905368264856e-13


In [53]:
accepted_elements[1]

17-element Vector{Tuple{CartesianIndex, Float64}}:
 (CartesianIndex(6, 6, 1, 1), 0.27829966247006294)
 (CartesianIndex(7, 7, 1, 1), 0.09470434129000373)
 (CartesianIndex(1, 1, 7, 6), 0.09354758031136134)
 (CartesianIndex(1, 6, 2, 6), 0.08600157766200227)
 (CartesianIndex(6, 2, 6, 1), 0.08581669689691288)
 (CartesianIndex(8, 1, 1, 7), 0.028980255271322787)
 (CartesianIndex(1, 8, 7, 1), 0.028719889123761264)
 (CartesianIndex(3, 7, 1, 6), 0.01664369708487467)
 (CartesianIndex(6, 3, 7, 1), 0.016423785663595765)
 (CartesianIndex(4, 7, 1, 7), 0.0103070935980967)
 (CartesianIndex(7, 4, 7, 1), 0.01017887433097495)
 (CartesianIndex(9, 7, 6, 7), 0.006002543216986441)
 (CartesianIndex(7, 9, 7, 6), 0.005874680619774803)
 (CartesianIndex(10, 1, 1, 8), 0.005460004790057394)
 (CartesianIndex(1, 10, 8, 1), 0.005410921655118967)
 (CartesianIndex(1, 6, 5, 6), 0.002817139986428244)
 (CartesianIndex(6, 5, 6, 1), 0.002734774054598103)

In [59]:
[A[1].sects[(0,0,0,0)][1,1,1,k] for k in 1:5]

5-element Vector{Float64}:
  0.522485325196873
 -0.033576763776410355
  1.9616523475617108e-5
  0.0009738712889111517
 -0.0014535226288984624

In [60]:
[A[1].sects[(0,0,0,0)][1,1,k,1] for k in 1:5]

5-element Vector{Float64}:
  0.522485325196873
 -0.03394577484519207
  8.89451283973256e-6
  0.0010640011018901601
 -0.0017764333291880657

In [61]:
[A[1].sects[(0,1,0,1)][1,1,1,k] for k in 1:5]

5-element Vector{Float64}:
 0.20430388454118534
 4.7977025223324214e-5
 0.02394000496786203
 0.0010356523081185322
 8.630948099597806e-7